## CH 05. 오차역전파법
<복습>
- 앞 장까지는 신경망 학습에 대해서 공부함
- 신경망의 가중치 매개변수에 대한 손실 함수의 기울기를 구하기 위해서 미분을 진행함
- 수치 미분법의 장단점
    - 장 ) 구현이 쉽고 간단함
    - 단 ) 기울기를 구하는 계산 시간이 오래걸림
- 📍 **오차역전파법 : 기울기를 효율적으로 계산하는 방법**

##### ⭐️ **5-1 ~ 5-3까지의 자세한 공부 내용은 벨로그에 정리해두었습니다.**

https://velog.io/@songcod_404_/posts



### 5-4. 단순한 계층 구현하기
사과 쇼핑의 예를 파이썬으로 구현
- 계산 그래프의 곱셈 노드를 'MulLayer'로 표현
- 계산 그래프의 덧셈 노드를 'AddLayer'로 표현

In [1]:
class MulLayer:
    def __init__(self):
        self.x = None
        self.y = None

    def forward(self, x, y):
        self.x = x
        self.y = y
        out = x*y
        
        return out

    def backward(self, dout):
        dx = dout * self.y
        dy = dout * self.x

        return dx, dy

In [2]:
apple = 100
apple_num = 2
tax = 1.1

mul_apple_layer = MulLayer()
mul_tax_layer = MulLayer()

apple_price = mul_apple_layer.forward(apple, apple_num)
price = mul_tax_layer.forward(apple_price, tax)

print(price)

220.00000000000003


In [3]:
dprice = 1

dapple_price, dtax = mul_tax_layer.backward(dprice)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)

print(dapple, dapple_num, dtax)

2.2 110.00000000000001 200


In [4]:
class AddLayer:
    def __init__(self):
        pass

    def forward(self, x, y):
        out = x + y
        return out
    
    def backward(self, dout):
        dx = dout * 1
        dy = dout * 1

        return dx, dy

In [5]:
apple = 100
apple_num = 2
orange = 150
orange_num = 3
tax = 1.1

#계층들
mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()

#순전파 계산
apple_price = mul_apple_layer.forward(apple, apple_num)
orange_price = mul_orange_layer.forward(orange, orange_num)
all_price = add_apple_orange_layer.forward(apple_price, orange_price)
price = mul_tax_layer.forward(all_price, tax)

#역전파 계산
dprice = 1
dall_price, dtax = mul_tax_layer.backward(dprice)
dapple_price, dorange_price = add_apple_orange_layer.backward(dall_price)
dorange, dorange_num = mul_orange_layer.backward(dorange_price)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)

print(price)
print(dapple_num, dapple, dorange, dorange_num, dtax)

715.0000000000001
110.00000000000001 2.2 3.3000000000000003 165.0 650


### 5-5. 활성화 함수 계층 구현하기
- ReLU와 시그모이드(Sigmoid) 함수 계층 구현하기

In [ ]:
class ReLU:
    def __init__(self):
        self.mask = None
    
    def forward(self, x): #입력 x가 넘파이 배열일 때... [0.1, -0.4, 0.8, -0.2]
        self.mask = (x <= 0) #x의 값이 0 이하인 것들은 True로 기억
        out = x.copy() #원본 배열 x를 copy해서
        out[self.mask] = 0 #0이하의 값들은 0으로 만듬 [0.1, 0, 0.8, -0.2]
 
        return out
    
    def backward(self, dout):
        dout[self.mask] = 0 #순전파 때 0이였던 곳은 기울기도 0
        dx = dout

        return dx

In [7]:
import numpy as np
x = np.array([[1.0, -0.5], [-2.0, 3.0]])
print(x)

mask = (x <= 0)
print(mask)

[[ 1.  -0.5]
 [-2.   3. ]]
[[False  True]
 [ True False]]


### 5-6. Affine/Softmax 계층 구현하기
#### 5-6-1. Affine 계층
- Affine 계층 : 신경망의 순전파 때 수행하는 행렬의 곱은 기하학에서는 '어파인 변환' 이라고 부르고, 이러한 행렬들의 곱을 수행하는 계층(np.dot(x, W1)...)을 어파인 계층이라고 부름

#### 5-6-2. 배치용 Affine 계층
- 데이터 N개를 묶어서 순전파하는 경우(배치로 묶어서 전파하는 경우)에 대한 생각을 해보자.
- 행렬 계산의 역전파는 크게 다를바가 없지만, **편향의 역전파에 주의하자.**
    - 순전파 때의 편향 덧셈은 X﹒W에 대한 편향이 각 데이터에 더해짐. 예를들어 N = 2인 경우 편향은 그 두 데이터에 각각 더해짐.
    - b = [b1, b2, b3]고 X﹒W = [[1,2,3],[4,5,6]]이라면... X﹒W + b = [[1+b1, 2+b2, 3+b3],[4+b1, 5+b2, 6+b3]]
    - 그렇기 때문에 역전파때도 각 데이터의 역전파 값이 편향의 원소에 모여야함(합이 되어야함)

In [8]:
X_dot_W = np.array([[0,0,0],[10,10,10]])
B = np.array([1,2,3])

print(X_dot_W)
print(X_dot_W + B)

[[ 0  0  0]
 [10 10 10]]
[[ 1  2  3]
 [11 12 13]]


In [10]:
dY = np.array([[1,2,3],[4,5,6]])
print(dY)

dB = np.sum(dY, axis = 0) #열기준 합
print(dB)

[[1 2 3]
 [4 5 6]]
[5 7 9]


In [11]:
class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        out = np.dot(x,self.W) + self.b

        return out
    
    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis = 0)

        return dx

#### 5-6-3. Softmax-with-Loss 계층


In [15]:
import sys, os
sys.path.append('..')
from common.functions import softmax, cross_entropy_error

class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None #손실함수 값
        self.y = None #softmax의 출력값
        self.t = None #레이블값(원-핫 벡터)

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)

        return self.loss
    
    def backward(self, dout = 1):
        batch_size = self.t.shape[0] #t=원-핫 레이블이기 때문에 (배치사이즈, 결과의 갯수)로 구성
        dx = (self.y - self.t) / batch_size  

        return dx


### 5-7. 오차역전파법 구현하기
#### 5-7-2. 오차역전파법을 적용한 신경망 구현하기

In [21]:
import sys, os
sys.path.append('..')
import numpy as np
from common.layers import *
from common.gradient import numerical_gradient
from collections import OrderedDict #순서를 기억하는 dict

class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        #가중치 초기화
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

        #계층 생성
        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['ReLU1'] = ReLU()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])
        self.last_layers = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)

        return x
    
    def loss(self, x, t):
        y = self.predict(x)
        
        return self.last_layers.forward(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis = 1)
        if t.ndim != 1 : t = np.argmax(t, axis = 1)

        accuracy = np.sum(y==t) / float(x.shape[0])

        return accuracy
    
    def numerical_gradient(self, x, t):
        loss_W = lambda W : self.loss(x, t)

        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

        return grads
    
    def gradient(self, x, t):
        self.loss(x, t)

        dout = 1
        dout = self.layers.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        #결과 저장
        grads = {}
        grads['W1'] = self.layers['Affine1'].dW
        grads['b1'] = self.layers['Affine1'].db
        grads['W2'] = self.layers['Affine2'].dW
        grads['b2'] = self.layers['Affine2'].db

        return grads


#### 5-7-3. 오차역전파법으로 구한 기울기 검증하기


In [22]:
import sys, os
sys.path.append('..')
import numpy as np
from dataset.mnist import load_mnist
from two_layer_net import TwoLayerNet

(x_train, t_train),(x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

x_batch = x_train[:3]
t_batch = t_train[:3]

grad_numerical = network.numerical_gradient(x_batch, t_batch)
grad_backprop = network.gradient(x_batch, t_batch)

for key in grad_numerical.keys():
    diff = np.average(np.abs(grad_backprop[key] - grad_numerical[key]))
    print(key + ":" + str(diff))

W1:2.021120190420382e-10
b1:1.0250740606430191e-09
W2:7.374810122695585e-08
b2:1.4710800973483097e-07


#### 5-7-4. 오차역전파법을 사용한 학습 구현하기
- 기울기를 오차역전파법을 이용해서 구함

In [23]:
import sys, os
sys.path.append('..')
import numpy as np
from dataset.mnist import load_mnist
from two_layer_net import TwoLayerNet

#데이터 읽기
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)
network = TwoLayerNet(input_size = 784, hidden_size= 50, output_size=10)

iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]

    grad = network.gradient(x_batch, t_batch)

    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate*grad[key]

    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)

    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)

        print(train_acc, test_acc)


0.10113333333333334 0.1002
0.7920833333333334 0.7964
0.8780333333333333 0.8811
0.8998 0.9033
0.9084666666666666 0.911
0.9156333333333333 0.919
0.9199 0.9227
0.9240166666666667 0.9257
0.9275 0.9297
0.9311 0.9331
0.9333833333333333 0.9331
0.93615 0.9349
0.9384333333333333 0.9381
0.9406333333333333 0.9395
0.9422833333333334 0.9419
0.9451166666666667 0.9452
0.9465166666666667 0.9465
